# Thử nghiệm truy xuất bảng từ database cho pipeline Text2SQL

**Pipeline:**

1. Chia schema thành từng document chứa: tên bảng, mô tả, cột, quan hệ  
2. Embed từng document thành vector (multilingual-e5-large hoặc jina-v4)  
3. Lưu trữ vector trong Qdrant  
4. Embed câu hỏi người dùng  
5. Dùng vector câu hỏi để tìm top-K schema gần nhất  
6. Dùng LLM để chọn ra các bảng liên quan từ top-K  

**Các model thử nghiệm:**
- multilingual-e5-large: prefix vs không prefix  
- jina-v4: single-vector vs multi-vector  
- LLM: gpt-oss-20b và gpt-oss-120b


In [1]:
# ====== LOAD SCHEMA ======
from pathlib import Path

SCHEMA_FILE = Path("pipeline/vi_schema.txt")
raw = SCHEMA_FILE.read_text(encoding="utf-8")

# Each table block is separated by  "=== TABLE ==="
docs = [d.strip() for d in raw.split("=== TABLE ===") if d.strip()]

print("Number of schema documents:", len(docs))
print("\n--- First schema document preview ---\n")
print(docs[0][:1500])


Number of schema documents: 20

--- First schema document preview ---

Table: khach_hang
Description: Lưu thông tin khách hàng (tài khoản người dùng) trong hệ thống thương mại điện tử.
Schema:
- khach_hang_id UUID
- ho_ten TEXT
- email TEXT
- so_dien_thoai TEXT
- ngay_sinh DATE
- gioi_tinh TEXT
- trang_thai TEXT
- ngay_tao TIMESTAMP
- ngay_cap_nhat TIMESTAMP
Keys:
- Primary Key: khach_hang_id
- Unique: email
- Unique: so_dien_thoai


In [2]:
# ===== EXTRACT TABLE NAME =====
import re

def extract_table_name(doc: str) -> str:
    pattern = r"^Table:\s*([a-zA-Z0-9_]+)"
    m = re.search(pattern, doc, flags=re.MULTILINE)
    if m:
        return m.group(1).strip()
    return "UNKNOWN_TABLE"

table_names = [extract_table_name(d) for d in docs]

print("Table names extracted:", len(set(table_names)))
print("First 10 names:", table_names[:10])


Table names extracted: 20
First 10 names: ['khach_hang', 'dia_chi_khach_hang', 'danh_muc_san_pham', 'thuong_hieu', 'nha_ban', 'san_pham', 'bien_the_san_pham', 'kho', 'ton_kho', 'don_hang']


In [3]:
# ===== E5 EMBEDDING EXAMPLE FOR 1 DOC =====
from sentence_transformers import SentenceTransformer

e5_model = SentenceTransformer("intfloat/multilingual-e5-large")

doc = docs[0]

# Add "passsage" prefix for doc
vec_pref = e5_model.encode(
        f"passage: {doc}",
        normalize_embeddings=False,
    ).tolist()

vec_nopref = e5_model.encode(
        doc,
        normalize_embeddings=False,
    ).tolist()

print(f"Doc embedded with prefix: {vec_pref}")
print(f"Doc embedded without prefix: {vec_nopref}")



c:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Doc embedded with prefix: [0.05784235894680023, -0.01073427777737379, -0.022943058982491493, -0.03341913968324661, 0.031925663352012634, -0.056772008538246155, -0.01213338878005743, 0.0753345638513565, 0.05047885701060295, -0.008627274073660374, 0.051065340638160706, 0.018769698217511177, -0.0444033220410347, 0.00660029286518693, -0.028426848351955414, 0.00087665522005409, -0.02936103194952011, 0.03721155598759651, -0.011904975399374962, -0.010689489543437958, 0.03439394384622574, -0.0038464877288788557, -0.01605869270861149, -0.026027753949165344, -0.007820431143045425, -0.021696776151657104, -0.02122856117784977, -0.04072851687669754, 0.00042004953138530254, -0.015218853950500488, 0.013654148206114769, 0.010282229632139206, -0.029700692743062973, -0.02212502434849739, -0.03692842274904251, 0.02106194570660591, 0.030643602833151817, 0.014875506050884724, -0.04422958567738533, 0.044671427458524704, -0.015351453796029091, 0.04229972884058952, 0.0005295398295857012, -0.021480899304151535

In [4]:
# ===== E5 UPSERT A POINT INTO QDRANT EXAMPLE =====

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

QDRANT_URL = "http://localhost:6333"
client = QdrantClient(url=QDRANT_URL)

E5_COLLECTION_NAME_NOTEBOOK = "notebook_e5_example_collection"

# Delete collection if already existed
if client.collection_exists(E5_COLLECTION_NAME_NOTEBOOK):
    client.delete_collection(E5_COLLECTION_NAME_NOTEBOOK)

# Create a collection with 2 namespace to store prefix and non prefix vector
client.create_collection(
    collection_name=E5_COLLECTION_NAME_NOTEBOOK,
    vectors_config={
        # Prefix
        "pref": VectorParams(
            size=1024,
            distance=Distance.COSINE,
        ),
        # No prefix 
        "nopref": VectorParams(
            size=1024,
            distance=Distance.COSINE,
        ),
    },
)

# Create a Qdrant point using the vector and embedded doc as payload
e5_point = [PointStruct(
            id=1,
            vector={
                "pref": vec_pref,
                "nopref": vec_nopref,
            },
            payload={
                # Store full schema text 
                "doc": doc,
            },
        )]

client.upsert(
    collection_name=E5_COLLECTION_NAME_NOTEBOOK,
    points=e5_point,
)
print("Upserted a point")



Upserted a point


In [5]:
# ===== JINA EMBEDDING EXAMPLE FOR 1 DOC =====
from transformers import AutoModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
# trust_remote_code=True is required for jina-embeddings-v4 encode_text()
jina_model = AutoModel.from_pretrained("jinaai/jina-embeddings-v4", trust_remote_code=True).to(device)

single_vec = jina_model.encode_text(
        doc,
        task="retrieval",
        truncate_dim=1024,
        max_length=512,
    )

multi_vec = jina_model.encode_text(
        doc,
        task="retrieval",
        return_multivector=True,
        max_length=512,
    )

print(f"Doc embedded as a single vector: {single_vec}")
print(f"Doc embedded as multi vector: {multi_vec}")

`torch_dtype` is deprecated! Use `dtype` instead!
Encoding texts...: 100%|██████████| 1/1 [00:16<00:00, 16.09s/it]

Doc embedded as a single vector: tensor([ 0.0432, -0.0032,  0.0583,  ...,  0.0008, -0.0013, -0.0164],
       dtype=torch.bfloat16)
Doc embedded as multi vector: tensor([[-0.0820,  0.1074,  0.0811,  ..., -0.0613,  0.1309,  0.0132],
        [ 0.0325,  0.1187,  0.1338,  ..., -0.0569,  0.0369,  0.0376],
        [-0.0398, -0.0164,  0.1865,  ..., -0.0540,  0.0571,  0.0635],
        ...,
        [-0.0498, -0.2168,  0.0845,  ...,  0.1069,  0.0476,  0.0540],
        [-0.0593, -0.1426,  0.0889,  ...,  0.1777,  0.1484,  0.1133],
        [-0.0723, -0.1006,  0.0640,  ...,  0.1089,  0.0962, -0.0679]],
       dtype=torch.bfloat16)


In [6]:
# ===== JINA UPSERT A POINT INTO QDRANT EXAMPLE =====

from qdrant_client.models import Distance, VectorParams, PointStruct, MultiVectorConfig, MultiVectorComparator

JINA_COLLECTION_NAME_NOTEBOOK = "notebook_jina_example_collection"

# Delete collection if already existed
if client.collection_exists(JINA_COLLECTION_NAME_NOTEBOOK):
    client.delete_collection(JINA_COLLECTION_NAME_NOTEBOOK)

# Create a collection with 2 namespace to store single and multi vector
client.create_collection(
    collection_name=JINA_COLLECTION_NAME_NOTEBOOK,
    vectors_config={
        "single": VectorParams(size=1024, distance=Distance.COSINE),
        "multi": VectorParams(
            size=128,
            distance=Distance.COSINE,
            multivector_config=MultiVectorConfig(comparator=MultiVectorComparator.MAX_SIM),
        ),
    },
)

# jina-embedding-v4 returns embedded text as Pytorch Tensors
# Qdrant can't store Pytorch Tensors so we need to convert to python list
single_vec_list = single_vec.detach().float().cpu().numpy().tolist()
multi_vec_list = multi_vec.detach().float().cpu().numpy().tolist()

jina_point = [PointStruct(
            id=1,
            vector={
                "single": single_vec_list,
                "multi": multi_vec_list,
            },
            payload={"doc": doc},
        )]

client.upsert(collection_name=JINA_COLLECTION_NAME_NOTEBOOK, points=jina_point)
print("Upserted point to qdrant")




Upserted point to qdrant


In [7]:
#===== RUN E5 EMBEDDING =====
import runpy
import os

"""
Run actual e5 ingest table in the pipeline for the whole schema
"""
runpy.run_path("pipeline/ingest_table_e5.py", run_name="__main__")

Parsed table docs: 20
Upserted 20 points into vi_schema_E5


{'__name__': '__main__',
 '__doc__': '\ningest_table.py\n\nPurpose:\n  - Ingest table schema documents\n  - Embed each table schema using multilingual-e5-large\n  - Store vectors in Qdrant for similarity search\n\nVector strategy:\n  - pref   : "passage: <doc>"  (query-time prefix enabled)\n  - nopref : "<doc>"           (no prefix)\n\n',
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': 'pipeline/ingest_table_e5.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _fr

In [8]:
#===== RUN JINA EMBEDDING =====
import runpy
"""
Run actual jina ingest table in the pipeline for the whole schema
"""
runpy.run_path("pipeline/ingest_table_jina.py", run_name="__main__")

Using device: cpu


Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 23967.45it/s]


Parsed table docs: 20


Encoding texts...: 100%|██████████| 1/1 [00:17<00:00, 17.26s/it]


Upserted 20 points into vi_schema_jina_v4


{'__name__': '__main__',
 '__doc__': '\nPurpose:\n  - Read schema docs from a text file (blocks separated by "=== TABLE ===")\n  - Embed each block using jina-embeddings-v4\n  - Store embeddings in Qdrant under 2 vector names:\n      1) "single": single-vector\n      2) "multi" : multi-vector \n',
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': 'pipeline/ingest_table_jina.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spe

In [9]:
# ===== EMBEDDING SEARCH WITH E5 EXAMPLE =====
import requests

# Use our actual collection that storing the whole schema vectors for e5
E5_COLLECTION_NAME = "vi_schema_E5"

query = "Top danh mục sản phẩm theo tổng doanh thu bán hàng"

# Add "query" prefix for query embedding
pref_query_vec = e5_model.encode(
        f"query: {query}",
        normalize_embeddings=False,
    ).tolist()

# No prefix
nopref_query_vec = e5_model.encode(
        query,
        normalize_embeddings=False,
    ).tolist()

# Search URL for request
e5_search_url = f"{QDRANT_URL}/collections/{E5_COLLECTION_NAME}/points/search"

# Payload for embed search with prefix
e5_pref_search_payload = {
        "vector": {
            "name": "pref",
            "vector": pref_query_vec,
        },
        "limit": 5,
        "with_payload": True,
    }

# No prefix payload
e5_nopref_search_payload = {
        "vector": {
            "name": "nopref",
            "vector": nopref_query_vec,
        },
        "limit": 5,
        "with_payload": True,
    }

# Requests
pref_request= requests.post(e5_search_url, json=e5_pref_search_payload, timeout=60)
nopref_request = requests.post(e5_search_url, json=e5_nopref_search_payload, timeout=60)

# Hits returned from qdrant search
pref_hits = pref_request.json()["result"]
nopref_hits = nopref_request.json()["result"]



def format_hits(hits):
    """Normalize Qdrant hits into stable output schema."""
    out = []
    for h in hits:
        payload = h.get("payload") or {}
        doc = payload.get("doc", "") or ""
        out.append(
            {
                "table": extract_table_name(doc),
                "score": float(h["score"]),
                "doc": doc,
            }
        )
    return out

pref_result = format_hits(pref_hits)
nopref_result = format_hits(nopref_hits)

print(f"Query search with prefix:\n {pref_result}")
print(f"Query search without prefix:\n {nopref_result}")


Query search with prefix:
 [{'table': 'san_pham', 'score': 0.8335432, 'doc': 'Table: san_pham\nDescription: Sản phẩm “mức cha” (SPU) do nhà bán đăng; SKU nằm ở bảng bien_the_san_pham.\nSchema:\n- san_pham_id UUID\n- nha_ban_id UUID\n- danh_muc_id UUID\n- thuong_hieu_id UUID\n- ten_san_pham TEXT\n- mo_ta TEXT\n- trang_thai TEXT\n- ngay_tao TIMESTAMP\n- ngay_cap_nhat TIMESTAMP\nKeys:\n- Primary Key: san_pham_id\n- Foreign Key: nha_ban_id -> nha_ban.nha_ban_id\n- Foreign Key: danh_muc_id -> danh_muc_san_pham.danh_muc_id\n- Foreign Key: thuong_hieu_id -> thuong_hieu.thuong_hieu_id\n- Index: nha_ban_id\n- Index: danh_muc_id\n- Index: thuong_hieu_id'}, {'table': 'danh_gia_san_pham', 'score': 0.83302975, 'doc': 'Table: danh_gia_san_pham\nDescription: Đánh giá sản phẩm theo SKU từ khách hàng đã mua (rating + review).\nSchema:\n- danh_gia_id UUID\n- sku_id UUID\n- khach_hang_id UUID\n- don_hang_id UUID\n- diem INT\n- noi_dung TEXT\n- ngay_tao TIMESTAMP\nKeys:\n- Primary Key: danh_gia_id\n- Fore

In [10]:
# ===== EMBEDDING SEARCH WITH JINA EXAMPLE =====

# Use our actual collection that storing the whole schema vectors for jina
JINA_COLLECTION_NAME = "vi_schema_jina_v4"
query_single_vec = jina_model.encode_text(
        query,
        task="retrieval",
        truncate_dim=1024,
        max_length=512,
    ).detach().float().cpu().numpy().tolist()

query_multi_vec = jina_model.encode_text(
        query,
        task="retrieval",
        return_multivector=True,
        max_length=512,
    ).detach().float().cpu().numpy().tolist()

# 
single_vec_hit = client.query_points(
        collection_name=JINA_COLLECTION_NAME,
        query=query_single_vec,
        using="single",
        limit=5,
        with_payload=True,
    )

multi_vec_hit = client.query_points(
        collection_name=JINA_COLLECTION_NAME,
        query=query_multi_vec,
        using="multi",
        limit=5,
        with_payload=True,
    )


print(f"Single vector embedding search result: \n{single_vec_hit.points}")
print(f"Multi vector embedding search result: \n{multi_vec_hit.points}")




Encoding texts...: 100%|██████████| 1/1 [00:02<00:00,  2.47s/it]

Single vector embedding search result: 
[ScoredPoint(id=2, version=1, score=0.59376246, payload={'doc': 'Table: danh_muc_san_pham\nDescription: Phân loại sản phẩm theo danh mục (hỗ trợ cây danh mục bằng danh_muc_cha_id).\nSchema:\n- danh_muc_id UUID\n- danh_muc_cha_id UUID\n- ten_danh_muc TEXT\n- mo_ta TEXT\n- trang_thai TEXT\n- ngay_tao TIMESTAMP\nKeys:\n- Primary Key: danh_muc_id\n- Foreign Key: danh_muc_cha_id -> danh_muc_san_pham.danh_muc_id (self-reference)\n- Index: danh_muc_cha_id'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=5, version=1, score=0.58403367, payload={'doc': 'Table: san_pham\nDescription: Sản phẩm “mức cha” (SPU) do nhà bán đăng; SKU nằm ở bảng bien_the_san_pham.\nSchema:\n- san_pham_id UUID\n- nha_ban_id UUID\n- danh_muc_id UUID\n- thuong_hieu_id UUID\n- ten_san_pham TEXT\n- mo_ta TEXT\n- trang_thai TEXT\n- ngay_tao TIMESTAMP\n- ngay_cap_nhat TIMESTAMP\nKeys:\n- Primary Key: san_pham_id\n- Foreign Key: nha_ban_id -> nha_ban.nha_ban_id\n- Forei

## Độ chính xác của model embedding

### Dữ liệu đánh giá:  
"vi-test-set" bao gồm:
- Câu hỏi bằng tiếng việt 
- Danh sách bảng cần dùng để trả lời câu hỏi

### Tiêu chí đánh giá:
Recall@k:
- Recall = Số bảng nằm trong candidate cũng nằm trong bảng cần dùng / Số bảng cần dùng
- k là số bảng candidate trả về từ embedding search
- Tính Recall@3, Recall@5, Recall@10 cho mỗi query
- Tính Recall@3, Recall@5, Recall@10 trung bình cho mỗi cấu hình của mỗi model

### Các model và cấu hình được đánh giá:
**Multilingual-e5-model**:    
- Prefix  
- Không prefix

**Jina-v4**:  
- Single vector  
- Multi vector  


In [11]:
# ==== TEST SET ==== 
from e5_retrieval_accuracy import parse_test_set

TEST_SET_PATH = "vi_test_set.txt"
test_set = parse_test_set(TEST_SET_PATH)
print(test_set)


[('Tổng doanh thu theo từng nhà bán trong quý trước', ['nha_ban', 'san_pham', 'bien_the_san_pham', 'chi_tiet_don_hang', 'don_hang']), ('Top nhà bán theo doanh thu ròng sau khi trừ hoàn tiền', ['nha_ban', 'san_pham', 'bien_the_san_pham', 'chi_tiet_don_hang', 'don_hang', 'tra_hang_hoan_tien']), ('Tổng số tiền đã thanh toán theo phương thức thanh toán mỗi tháng', ['thanh_toan']), ('Chênh lệch giữa tổng đơn hàng và số tiền đã thanh toán theo từng đơn', ['don_hang', 'thanh_toan']), ('Tác động của khuyến mãi lên số lượng đơn hàng', ['don_hang', 'ap_dung_khuyen_mai', 'khuyen_mai']), ('Doanh thu tạo ra từ các sản phẩm được khuyến mãi', ['ap_dung_khuyen_mai', 'don_hang', 'chi_tiet_don_hang', 'bien_the_san_pham']), ('Giá trị giảm giá trung bình theo từng mã khuyến mãi', ['khuyen_mai', 'ap_dung_khuyen_mai']), ('Top danh mục sản phẩm theo tổng doanh thu bán hàng', ['danh_muc_san_pham', 'san_pham', 'bien_the_san_pham', 'chi_tiet_don_hang', 'don_hang']), ('Tỉ lệ hoàn tiền theo danh mục sản phẩm', ['

In [12]:
#===== RUN E5 accuracy script =====
import runpy

runpy.run_path("e5_retrieval_accuracy.py", run_name="__main__") # Output to results/e5_retrieval_accuracy.txt


{'__name__': '__main__',
 '__doc__': '\nE5-based table retrieval accuracy on 2 modes:\n - pref    : query-prefixed retrieval ("query: ...")\n - nopref  : raw query retrieval\n',
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': 'e5_retrieval_accuracy.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'),
  '__build_class__':

In [13]:
#===== Read E5 accuracy script =====
from pathlib import Path

path = Path("results/e5_retrieval_accuracy.txt")

print(path.read_text(encoding="utf-8"))


NUM QUERIES: 18
K_VALUES: [1, 3, 5, 10]
--------------------------------------------------------------------------------
[1/18] QUERY: Tổng doanh thu theo từng nhà bán trong quý trước
GOLD TABLES (5): nha_ban, san_pham, bien_the_san_pham, chi_tiet_don_hang, don_hang

[PREF] Retrieved tables:
   1. don_hang                  score=0.8314 IN
   2. nha_ban                   score=0.8182 IN
   3. danh_gia_san_pham         score=0.8140
   4. chi_tiet_don_hang         score=0.8134 IN
   5. dia_chi_khach_hang        score=0.8132
   6. san_pham                  score=0.8119 IN
   7. thanh_toan                score=0.8097
   8. van_chuyen                score=0.8081
   9. tra_hang_hoan_tien        score=0.8072
  10. ap_dung_khuyen_mai        score=0.8064
[PREF] Recall@1 : 0.200
[PREF] Recall@3 : 0.400
[PREF] Recall@5 : 0.600
[PREF] Recall@10: 0.800

[NO-PREF] Retrieved tables:
   1. don_hang                  score=0.8293 IN
   2. nha_ban                   score=0.8219 IN
   3. san_pham          

In [14]:
#===== RUN JINA ACCURACY SCRIPT =====
import runpy

runpy.run_path("jina_retrieval_accuracy.py", run_name="__main__") # Output to results/jina_retrieval_accuracy.txt


Encoding texts...: 100%|██████████| 1/1 [00:02<00:00,  2.25s/it]


{'__name__': '__main__',
 '__doc__': '\nEvaluate jina-v4-embedder retrieval accuracy on 2 mode: single and multi vectors\n\nThis script:\n  - Calls get_candidate_tables(query) that retrieve table using single vector\n  - Calls get_candidate_multi_vector(query) that retrieve using multi vector\n  - Use Recall@K\n  - Reports per-query Recall@K and overall\n',
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': 'jina_retrieval_accuracy.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
 

In [15]:
#===== READ JINA ACCURACY SCRIPT =====
from pathlib import Path

path = Path("results/jina_retrieval_accuracy.txt")

print(path.read_text(encoding="utf-8"))

NUM QUERIES: 18
K_VALUES: [1, 3, 5, 10]
--------------------------------------------------------------------------------
[1/18] QUERY: Tổng doanh thu theo từng nhà bán trong quý trước
GOLD TABLES (5): nha_ban, san_pham, bien_the_san_pham, chi_tiet_don_hang, don_hang

[SINGLE] Retrieved tables:
   1. nha_ban                   score=0.5026 IN
   2. san_pham                  score=0.4714 IN
   3. thanh_toan                score=0.4693
   4. thuong_hieu               score=0.4154
   5. ton_kho                   score=0.4101
   6. don_hang                  score=0.4047 IN
   7. chi_tiet_don_hang         score=0.4000 IN
   8. chi_tiet_tra_hang         score=0.3912
   9. dia_chi_khach_hang        score=0.3881
  10. danh_muc_san_pham         score=0.3870
[SINGLE] Recall@1 : 0.200
[SINGLE] Recall@3 : 0.400
[SINGLE] Recall@5 : 0.400
[SINGLE] Recall@10: 0.800

[MULTI] Retrieved tables:
   1. san_pham                  score=7.0842 IN
   2. nha_ban                   score=6.7038 IN
   3. thanh_toan

## Phân tích hiệu năng truy xuất bảng của các mô hình embedding

## 1. Multilingual-E5-Large: Prefix vs Không Prefix

### Nhận xét
- Dựa vào output của mỗi query và recall@1 trung bình, 2 cấu hình đều cho bảng top 1 giống nhau 
- Recall@3 và Recall@10 cho thấy việc sử dụng prefix cải thiện độ chính xác của embedding search
- Recall@5 trung bình của không prefix cao hơn một chút so với có prefix

### Kết luận
- Dựa vào kết quả, việc sử dụng prefix có cải thiện độ chính xác. 
- Điều này cũng được nhắc đến bởi tác giả của model là model được huấn luyện dựa trên việc sử dụng prefix, không sử dụng prefix sẽ làm giảm độ chính xác

## 2. Jina Embeddings v4: Single-Vector vs Multi-Vector

### Nhận xét
- Recall@1 của multi-vector cao hơn so với single-vector -> khả năng trả về bảng liên quan ở top 1 của multi-vector tốt hơn single-vector
-  Recall@3,Recall@5 và Recall@10 của single vector cao hơn so với multi-vector -> cấu hình single-vector trả về bảng liên quan tốt hơn so với multi-vector khi k càng tăng

### Kết luận 
- Recall@1 của multi-vector không mang lại giá trị khi có tỉ lệ recall khá thấp
- Single-vector là lựa chọn tối ưu hơn khi cho tỉ lệ recall tốt hơn

## 3. Jina Embeddings v4 vs Multilingual-E5-Large

### Nhận xét
- Recall@10 của e5 có prefix có tỉ lệ recall cao nhất
- Với k là 3 và 5, e5 đều có tỉ lệ recall cao hơn so với jina
- Jina chỉ vượt trội hơn e5 ở recall@1

### Kết luận 
- Recall@1 của jina cũng không có nhiều giá trị khi có tỉ lệ recall thấp
- Trong phần lớn trường hợp, với những schema có số lượng table cao (>10) e5 có prefix là lựa chọn tối ưu nhất khi có tỉ lệ recall cao nhất
- Có thể xem xét sử dụng e5 không prefix cho những schema có số lượng table thấp (<10) do có recall@5 cao nhất.


## Chọn bảng liên quan bằng LLM

### Pipeline:
1. Lấy tên bảng candidate từ embedding search (dùng e5 có prefix)
2. Lấy mô tả, primary key và foregin key từ schema cho những bảng tương ứng trong candidate
3. Tạo prompt người dùng bằng hướng dẫn sử dụng bảng và tên bảng đi kèm với mô tả và keys
4. Gọi LLM bằng prompt người dùng và chỉ dẫn của hệ thống (system instruction)
5. Kiểm tra tính hợp lệ của output của LLM (chỉ được trả về json {"final table": [table1,....]}, không SQL, không được chứa bảng ngoài candidate)
6. Format lại output nếu hợp lệ


In [16]:
#===== GET TABLE NAMES FROM CANDIDATES =====
from pipeline.get_candidate_e5 import get_candidate_pref

# Use e5 with prefix as it's having the highest recall
query = test_set[0][0]
cand = get_candidate_pref(query, 10).get("candidates")

cand_tables = [c["table"] for c in cand if c.get("table")]
print(cand_tables)


['don_hang', 'nha_ban', 'danh_gia_san_pham', 'chi_tiet_don_hang', 'dia_chi_khach_hang', 'san_pham', 'thanh_toan', 'van_chuyen', 'tra_hang_hoan_tien', 'ap_dung_khuyen_mai']


In [17]:
#===== EXTRACT TABLE DESCRIPTION, PRIMARY KEY AND FOREGIN KEY FROM DOC =====
from pipeline.get_final import extract_description_from_doc, extract_pk_fk_from_doc

tables_for_template = []
for c in cand:
    doc = c.get("doc")
    tables_for_template.append({
        "name": c["table"],
        "description": extract_description_from_doc(doc),
        "keys": extract_pk_fk_from_doc(doc)
    })
print(tables_for_template[0])


Using gpt-oss-120b
USE_INSTRUCTION: true
{'name': 'don_hang', 'description': 'Đơn hàng của khách hàng; tổng tiền được tính từ chi_tiet_don_hang + phí/giảm giá.', 'keys': '- Primary Key: don_hang_id\n- Foreign Key: khach_hang_id -> khach_hang.khach_hang_id\n- Foreign Key: dia_chi_giao_id -> dia_chi_khach_hang.dia_chi_id'}


In [18]:
#===== BUILD USER PROMPT =====
from pipeline.get_final import render_template_vi, load_template_text, load_table_usage_instructions
print(load_table_usage_instructions)

# Add table usage and candidate tables with description and relationships into the user query template
user_prompt = render_template_vi(
    template_text=load_template_text(),
    query=query,
    tables=tables_for_template,
    table_usage_instructions=load_table_usage_instructions()
)

print(user_prompt)

<function load_table_usage_instructions at 0x000002284ACFEDE0>
═══════════════════════════════════════════════════════════════════════════════
                               YOUR TASK
═══════════════════════════════════════════════════════════════════════════════

You are a TABLE SELECTION SYSTEM in the Text-to-SQL pipeline.

OBJECTIVE:
Select ALL tables that could be needed to write a complete SQL query that answers the question.
When in doubt, INCLUDE the table rather than exclude it.

MANDATORY CONSTRAINTS:
✓ ONLY select tables from the list provided below
✓ DO NOT create new table names
✓ DO NOT output SQL queries
✓ DO NOT provide explanations or comments

TABLE SELECTION RULES:
1. Prioritize rules: If the question MATCHES "TABLE USAGE RULES" below
   → Use the tables listed in that rule
   → Consider adding related tables for complete JOIN paths

2. Multiple rules match:
   → Combine tables from all matching rules

3. No matching rules: If no rules match
   → Select tables based o

In [20]:
#===== LOAD SYSTEM INSTRUCTION =====
from pipeline.instructions.system_prompt_vi import SYSTEM_INSTRUCTION

print("System Instruction:")
print(SYSTEM_INSTRUCTION)

System Instruction:

# VAI TRÒ
Bạn là trợ lý chọn bảng cho hệ thống cơ sở dữ liệu.

# MỤC TIÊU
Chọn TẤT CẢ các bảng có thể cần thiết để viết câu truy vấn SQL hoàn chỉnh trả lời câu hỏi.
- Khi không chắc chắn, HÃY BAO GỒM bảng thay vì loại bỏ
- Bao gồm các bảng cho đường dẫn JOIN ngay cả khi chúng có vẻ gián tiếp

# RÀNG BUỘC
1. CHỈ chọn bảng từ danh sách bảng được cung cấp
2. KHÔNG tạo tên bảng mới
3. KHÔNG viết SQL
4. KHÔNG giải thích

# QUY TRÌNH LÀM VIỆC

BƯỚC 1: Phân tích câu hỏi
- Xác định dữ liệu mà câu hỏi cần
- Xác định tất cả các thực thể và mối quan hệ liên quan

BƯỚC 2: Kiểm tra quy tắc sử dụng bảng
- Xem xét các QUY TẮC SỬ DỤNG BẢNG được cung cấp trong prompt
- Nếu có quy tắc phù hợp → Sử dụng các bảng được liệt kê VÀ xem xét thêm các bảng liên quan
- Nếu có nhiều quy tắc phù hợp → Kết hợp bảng từ tất cả các quy tắc

BƯỚC 3: Nếu không có quy tắc phù hợp, xác định bảng thủ công
- Bắt đầu với bảng thực thể chính dựa trên mô tả bảng
- Thêm tất cả các bảng có thể liên quan dựa 

In [21]:
#===== CALL LLM WITH QUERY EXAMPLE =====
import requests
import json
import os
from dotenv import load_dotenv

load_dotenv("pipeline/.env")

LLM_BASE_URL = os.getenv("LLM_BASE_URL", "https://mkp-api.fptcloud.com/v1")
LLM_MODEL = os.getenv("LLM_MODEL", "gpt-oss-120b")
LLM_API_KEY = os.getenv("LLM_API_KEY")

# LLM base URL (the same for both model)
url = f"{LLM_BASE_URL.rstrip('/')}/chat/completions"
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {LLM_API_KEY}",
}

# LLM payload (the same for both model)
payload = {
    "model": LLM_MODEL,
    "messages": [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": user_prompt},
    ],
    "temperature": 0,
    "max_tokens": 1024,
    "top_p": 1,
}

print(f"Calling LLM: {LLM_MODEL}")
print(f"Query: {query}")
print(f"Candidate tables: {cand_tables}\n")

r = requests.post(url, headers=headers, json=payload, timeout=120)
r.raise_for_status()
resp = r.json()

msg = resp["choices"][0]["message"]
content = msg.get("content", "")
reasoning = msg.get("reasoning_content", "")

print(f"LLM Response:\n{content}")
if reasoning:
    print(f"\nReasoning:\n{reasoning}")

Calling LLM: gpt-oss-20b
Query: Tổng doanh thu theo từng nhà bán trong quý trước
Candidate tables: ['don_hang', 'nha_ban', 'danh_gia_san_pham', 'chi_tiet_don_hang', 'dia_chi_khach_hang', 'san_pham', 'thanh_toan', 'van_chuyen', 'tra_hang_hoan_tien', 'ap_dung_khuyen_mai']

LLM Response:
{"final_tables":["don_hang","chi_tiet_don_hang","bien_the_san_pham","san_pham","nha_ban"]}

Reasoning:
We need to choose tables for question: "Tổng doanh thu theo từng nhà bán trong quý trước" meaning total revenue per seller in previous quarter. Need tables: don_hang (orders), san_pham (product with seller), maybe chi_tiet_don_hang? But revenue is from don_hang total. To get seller, need join san_pham to get nha_ban_id. san_pham has nha_ban_id. So need san_pham. Also need maybe chi_tiet_don_hang? Not necessary if revenue is in don_hang. But to ensure correct mapping, maybe need chi_tiet_don_hang? But rule: "DOANH THU / GMV / SỐ LƯỢNG BÁN cần tên sản phẩm / gắn nhà bán / gắn danh mục" includes don_hang, c

In [22]:
#===== VALIDATE AND FORMAT LLM OUTPUT =====
from pipeline.get_final import validate_and_filter_final_tables, extract_json_object

# Parse JSON from LLM output
try:
    result = json.loads(content)
except json.JSONDecodeError:
    result = extract_json_object(content) or {"raw": content} # In case responses are not in json format

result["reasoning_content"] = reasoning

print("Parsed result:")
print(json.dumps(result, ensure_ascii=False, indent=2))

# Validate and filter tables
final_tables = validate_and_filter_final_tables(result, cand_tables) # Ensure the final tables are in the candidate list

print(f"\nValidated final tables: {final_tables}")
print(f"\nExpected tables (from test set): {test_set[0][1]}")

# Calculate recall
expected = set(test_set[0][1])
predicted = set(final_tables)
recall = len(expected & predicted) / len(expected) if expected else 0
precision = len(expected & predicted) / len(predicted) if predicted else 0

print(f"\nRecall: {recall:.3f}")
print(f"Precision: {precision:.3f}")

Parsed result:
{
  "final_tables": [
    "don_hang",
    "chi_tiet_don_hang",
    "bien_the_san_pham",
    "san_pham",
    "nha_ban"
  ],
  "reasoning_content": "We need to choose tables for question: \"Tổng doanh thu theo từng nhà bán trong quý trước\" meaning total revenue per seller in previous quarter. Need tables: don_hang (orders), san_pham (product with seller), maybe chi_tiet_don_hang? But revenue is from don_hang total. To get seller, need join san_pham to get nha_ban_id. san_pham has nha_ban_id. So need san_pham. Also need maybe chi_tiet_don_hang? Not necessary if revenue is in don_hang. But to ensure correct mapping, maybe need chi_tiet_don_hang? But rule: \"DOANH THU / GMV / SỐ LƯỢNG BÁN cần tên sản phẩm / gắn nhà bán / gắn danh mục\" includes don_hang, chi_tiet_don_hang, bien_the_san_pham, san_pham. But our question only needs revenue per seller. Could use don_hang and san_pham via chi_tiet_don_hang and bien_the_san_pham. But simpler: don_hang join chi_tiet_don_hang join b

## Đánh giá độ chính xác của LLM

### Tiêu chí đánh giá:
**Recall**: 
- Recall = Số bảng LLM chọn đúng / Số bảng cần dùng
- Đo lường khả năng tìm được tất cả các bảng cần thiết

**Precision**: 
- Precision = Số bảng LLM chọn đúng / Số bảng LLM chọn ra
- Đo lường độ chính xác của các bảng được chọn (tránh chọn bảng thừa)

### Các LLM được đánh giá:
- **gpt-oss-20b**: Model nhỏ hơn, nhanh hơn
- **gpt-oss-120b**: Model lớn hơn, có khả năng reasoning tốt hơn

In [42]:
#===== RUN LLM ACCURACY TEST - GPT-OSS-20B =====
import runpy
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv("pipeline/.env")

# Set model to gpt-oss-20b
os.environ["LLM_MODEL"] = "gpt-oss-20b"
os.environ["LLM_TEST_OUT"] = "results/oss_20b_accuracy.txt"



print("Running accuracy test with gpt-oss-20b...")
runpy.run_path("llm_table_accuracy.py", run_name="__main__")  # Output to results/oss-20b_accuracy_report.txt

Running accuracy test with gpt-oss-20b...


{'__name__': '__main__',
 '__doc__': '\nEvaluate table selection accuracy for the whole pipeline:\n  - Retrieval candidates (vector search)\n  - LLM final table selection\nMetrics:\n  - Recall\n  - Precision\nOutputs:\n  - Per query breakdown\n  - Overall averages\n',
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': 'llm_table_accuracy.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', load

In [43]:
#===== READ GPT-OSS-20B RESULTS =====
from pathlib import Path

path = Path("results/oss_20b_accuracy.txt")
print(path.read_text(encoding="utf-8"))

----------------------------------------------------------------------------------------------------
QUERY: Tổng doanh thu theo từng nhà bán trong quý trước
REQUIRED TABLES (5): nha_ban, san_pham, bien_the_san_pham, chi_tiet_don_hang, don_hang

CANDIDATE TABLES (TOP-10)
------------------------------------------------------------
01. don_hang                            score=0.8314 IN
02. nha_ban                             score=0.8182 IN
03. danh_gia_san_pham                   score=0.8140 
04. chi_tiet_don_hang                   score=0.8134 IN
05. dia_chi_khach_hang                  score=0.8132 
06. san_pham                            score=0.8119 IN
07. thanh_toan                          score=0.8097 
08. van_chuyen                          score=0.8081 
09. tra_hang_hoan_tien                  score=0.8072 
10. ap_dung_khuyen_mai                  score=0.8064 
------------------------------------------------------------

[ERROR] LLM failed: LLM_API_KEY is missing
LLM TABLES (0):

In [44]:
#===== RUN LLM ACCURACY TEST - GPT-OSS-120B =====
import runpy
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv("pipeline/.env")

# Set model to gpt-oss-120b
os.environ["LLM_MODEL"] = "gpt-oss-120b"
os.environ["LLM_TEST_OUT"] = "results/oss_120b_accuracy.txt"

print("Running accuracy test with gpt-oss-120b...")
runpy.run_path("llm_table_accuracy.py", run_name="__main__")  # Output to results/oss_120b_accuracy_report.txt

Running accuracy test with gpt-oss-120b...


{'__name__': '__main__',
 '__doc__': '\nEvaluate table selection accuracy for the whole pipeline:\n  - Retrieval candidates (vector search)\n  - LLM final table selection\nMetrics:\n  - Recall\n  - Precision\nOutputs:\n  - Per query breakdown\n  - Overall averages\n',
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': 'llm_table_accuracy.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', load

In [45]:
#===== READ GPT-OSS-120B RESULTS =====
from pathlib import Path

path = Path("results/oss_120b_accuracy.txt")
print(path.read_text(encoding="utf-8"))

----------------------------------------------------------------------------------------------------
QUERY: Tổng doanh thu theo từng nhà bán trong quý trước
REQUIRED TABLES (5): nha_ban, san_pham, bien_the_san_pham, chi_tiet_don_hang, don_hang

CANDIDATE TABLES (TOP-10)
------------------------------------------------------------
01. don_hang                            score=0.8314 IN
02. nha_ban                             score=0.8182 IN
03. danh_gia_san_pham                   score=0.8140 
04. chi_tiet_don_hang                   score=0.8134 IN
05. dia_chi_khach_hang                  score=0.8132 
06. san_pham                            score=0.8119 IN
07. thanh_toan                          score=0.8097 
08. van_chuyen                          score=0.8081 
09. tra_hang_hoan_tien                  score=0.8072 
10. ap_dung_khuyen_mai                  score=0.8064 
------------------------------------------------------------

[ERROR] LLM failed: LLM_API_KEY is missing
LLM TABLES (0):

## Phân tích hiệu năng LLM Selection

### Kết quả tổng quan

1. **Recall**: 
   - Cả hai model đều đạt recall cao và không chênh lệch nhiều (>92%)
   - Điều này cho thấy LLM có khả năng tìm ra hầu hết các bảng cần thiết
   - Test đang sử dụng e5-multiluingal large có prefix là embedder với recall@10 là 95%, cả 2 LLM cho kết quả có recall lớn hơn 92%


2. **Precision**: 
   - GPT-OSS-20B có precision cao hơn
   - Cho thấy GPT-OSS-20B có khả năng lọc bỏ các bảng không cần thiết tốt hơn
   - Giúp giảm số lượng bảng thừa trong kết quả cuối cùng

3. **Trade-off**: 
   - GPT-OSS-20B: Nhanh hơn, chi phí thấp và precision cao hơn
   - GPT-OSS-120B: Chậm hơn, chi phí cao hơn và precision thấp hơn

### Kết luận

- **GPT-OSS-20B** là lựa chọn tối ưu cho pipeline hiện tại
- Pipeline hoàn chỉnh (E5 prefix + GPT-OSS-20B) đạt ~92% recall và ~85% precision
- Tuy nhiên để tạo ra một câu SQL chính xác và không lỗi thì recall của cả pipeline phải đạt 100% vì nó cần chứa tất cả bảng cần thiết để join và trả lời câu hỏi
- Precision không cần thiết phải đạt 100% nhưng phải giữ ở một mức tốt để đảm bảo không có nhiều bảng không cần thiết (noise) ảnh hưởng đến việc viết query